In [1]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

/home/bonavina/Documentos/Github/etl_rpvmm/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7069.90it/s]


In [2]:
import pandas as pd

materials_df = pd.read_csv("/home/bonavina/Documentos/Github/etl_rpvmm/data/materiais_2025.csv")
articles_df = pd.read_csv("/home/bonavina/Documentos/Github/etl_rpvmm/RPVM_2026_10_05/outputs/alunos_artigos_scival.csv")

In [6]:
import unicodedata

import numpy as np


def name_key(value: str) -> str:
    text = unicodedata.normalize("NFKD", str(value))
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    return " ".join(text.casefold().split())


theses = (
    materials_df.assign(author_key=materials_df["autor"].map(name_key))
    .loc[:, ["autor", "author_key", "titulo", "ano"]]
    .drop_duplicates(subset=["author_key", "titulo"])
    .reset_index(drop=True)
)
articles = (
    articles_df.assign(
        author_key=articles_df["aluno"].map(name_key),
        primeiro_autor=articles_df["primeiro_autor"].eq(1),
    )
    .loc[:, ["aluno", "author_key", "artigo", "doi", "ano_publicacao_scopus", "primeiro_autor"]]
    .drop_duplicates(subset=["author_key", "doi"])
    .reset_index(drop=True)
)

theses["thesis_idx"] = np.arange(len(theses))
articles["article_idx"] = np.arange(len(articles))
pairs = theses.merge(articles, on="author_key", how="inner")

# all-MiniLM embeddings are L2-normalized, so cosine similarity is the dot product.
thesis_vectors = model.encode(theses["titulo"].tolist(), normalize_embeddings=True)
article_vectors = model.encode(articles["artigo"].tolist(), normalize_embeddings=True)

thesis_rows = thesis_vectors[pairs["thesis_idx"].to_numpy()]
article_rows = article_vectors[pairs["article_idx"].to_numpy()]
pairs["cosine_similarity"] = np.sum(thesis_rows * article_rows, axis=1)

similarity = (
    pairs.loc[:, ["autor", "titulo", "artigo", "doi", "ano", "ano_publicacao_scopus", "primeiro_autor", "cosine_similarity"]]
    .sort_values(["autor", "cosine_similarity"], ascending=[True, False])
    .reset_index(drop=True)
)
summary = (
    similarity.groupby("autor", as_index=False)
    .agg(
        n_articles=("doi", "nunique"),
        cosine_max=("cosine_similarity", "max"),
        cosine_mean=("cosine_similarity", "mean"),
    )
    .sort_values("cosine_max", ascending=False)
    .reset_index(drop=True)
)

matched_authors = set(pairs["author_key"])
unmatched_theses = theses.loc[~theses["author_key"].isin(matched_authors), "autor"].tolist()

similarity

,autor,titulo,artigo,doi,ano,ano_publicacao_scopus,primeiro_autor,cosine_similarity
0,Arthur Dantas Mangussi,A data-centric approach to missing data imputa...,Studying the robustness of data imputation met...,10.1016/j.cose.2025.104574,2025,2025,True,0.772250
1,Arthur Dantas Mangussi,A data-centric approach to missing data imputa...,Exploring the Influence of Missing Data Imputa...,10.1016/j.artint.2026.104559,2025,2026,True,0.623965
2,Arthur Dantas Mangussi,A data-centric approach to missing data imputa...,A Label Propagation Approach for Missing Data ...,10.1109/access.2025.3559772,2025,2025,False,0.608647
3,Arthur Dantas Mangussi,A data-centric approach to missing data imputa...,mdatagen: A python library for the artificial ...,10.1016/j.neucom.2025.129478,2025,2025,True,0.520064
4,Arthur Dantas Mangussi,A data-centric approach to missing data imputa...,Factors influencing the response to periodonta...,10.1590/1678-7757-2025-0211,2025,2025,False,0.189283
5,Emerson Yoshiaki Okano,Anomaly detection in time series : meta-learni...,Ranking Time Series Anomaly Detection Algorith...,10.1590/0101-7438.2026.046.00302988,2025,2026,True,0.806604
6,Emerson Yoshiaki Okano,Anomaly detection in time series : meta-learni...,An analytics framework for graph-based anomaly...,10.1016/j.health.2026.100447,2025,2026,True,0.566103
7,Filipe Loyola Lopes,Label propagation approach for missing data im...,A Label Propagation Approach for Missing Data ...,10.1109/access.2025.3559772,2025,2025,True,0.861273
8,Filipe Loyola Lopes,Label propagation approach for missing data im...,mdatagen: A python library for the artificial ...,10.1016/j.neucom.2025.129478,2025,2025,False,0.500319
9,Humberto Gimenes Macedo,Inexact first-order methods with general direc...,An inexact first-order descent method with gen...,10.1007/s40314-026-03794-2,2026,2026,True,0.795526


In [7]:
similarity.to_csv("thesis_articles_similarity.csv", index=False)

In [5]:
unmatched_theses

['Abreu Armando',
 'Alfredo Antonio Alencar Exposito de Queiroz',
 'Andre da Costa Silva',
 'Daniel Meireles Meira',
 'Eduardo Vargas Ferreira',
 'Guilherme Pisani',
 'Guilherme Ribeiro da Silva',
 'Jacqueline Komatzu Huayanca',
 'Lucas de Abreu Julião',
 'Lucas Queiroz Melo da Costa',
 'Marcos de Pinho Tavares Proença',
 'Paulo Victor Freitas Lopes',
 'Rafaela Veloso de Oliveira',
 'Victor de Sá Nunes']